# 538 — Alex-sequenced Tier 1 model + fit

**Purpose:** Walk `5-Manuscript/Alex_Tier1_Sequential_Model_Outline.md` in order. This notebook builds the **data map** (\(Y\), \(L\) proxies, \(A\), diagnostics) the same way as **`535`**, but drops heavy **CELL 4 EDA** charts — those stay in `535` to avoid duplicate figures.

**Run order:** EDIT BLANK (log) → **CELL 0** → **CELL 1** → **CELL 2** → **CELL 3** → *(later cells: bins → LPM / logit → \(L^*\))*. Each numbered CELL is a **markdown** header immediately followed by a **code** cell whose first line is `# CELL N — …`.

- **Reference panel:** `datasets/mbb/player_season_panel_530.csv` via `sports_pipeline.paths.panel_530_csv()` (refresh with **`530_sports_pipeline.ipynb`** when you need a new export).
- **Simulation / comparative statics:** `537_Sports_Simulation.ipynb` only.

In [ ]:
# EDIT BLANK — session log
##### EDIT BLANK ###### DISCARD
# session: 2026-05-15 12:49 — populate 538: CELL 0–3 (Alex scaffold; no 535 CELL4 EDA).


### CELL 0 — Config · `RUN_CELL*` · knobs for Alex pipeline

Switchboard for **538** only. Gates **CELL 1–3** (environment → panel → Tier 1 columns). Fitting cells (bins, LPM, logit, \(L^*\)) will use the same knobs; **no** `CELL4_*` EDA flags here — use **`535`** for ventile/bar diagnostic plots.

Re-run this code cell after changing any flag.

In [ ]:
# CELL 0 — Config · RUN_CELL* · knobs for Alex pipeline

# --- Run gates (re-run CELL 0 after changes) ---
RUN_CELL1 = True  # Environment · imports · CFG · panel path
RUN_CELL2 = True  # Load panel · perf · legacy poolq_loo
RUN_CELL3 = True  # Tier 1 mechanism columns (congestion_*)

# --- Panel / Tier 1 (consumed by CELL 2–3 and later fit cells) ---
PERF_METRIC = "ppm"

# Rows below this minutes threshold dropped before Tier 1 group stats; None = keep all.
MIN_MINUTES_TIER1 = None

# "quality" → L proxy path favors congestion_quality (Tier 1 Q); "crowding" → crowding cols.
PRIMARY_POOL_MODE = "quality"
COMPUTE_WEIGHTED_CROWDING = False

print(
    "538 CELL 0:",
    f"RUN_CELL1={RUN_CELL1}",
    f"RUN_CELL2={RUN_CELL2}",
    f"RUN_CELL3={RUN_CELL3}",
    f"PERF_METRIC={PERF_METRIC!r}",
    f"PRIMARY_POOL_MODE={PRIMARY_POOL_MODE!r}",
    f"COMPUTE_WEIGHTED_CROWDING={COMPUTE_WEIGHTED_CROWDING}",
)

## CELL 1 — Environment · imports · panel path

**Alex outline:** sets up **§4 Map model to data** — same `PipelineConfig` + `panel_530_csv()` contract as **`535`**, without pulling in CELL 4 plotting helpers.

- Run **CELL 0** first.
- For an exact fresh panel, export from **`530_sports_pipeline.ipynb`** before loading here.

In [ ]:
# CELL 1 — Environment · imports · panel path
from __future__ import annotations

if RUN_CELL1:
    from sports_pipeline import paths
    from sports_pipeline.panel_build import load_panel
    from sports_pipeline.config import PipelineConfig

    # Match 530 / 535 defaults; tighten binning in later fit cells if needed (Alex ladder starts descriptive).
    CFG = PipelineConfig(
        panel_season_min=2011,
        panel_season_max=2021,
        ventiles=20,
        poolq_binning="quantile",
        poolq_winsor_quantiles=(0.05, 0.95),
        perf_zscore_within_season=True,
        restrict_teams_by_draftees=False,
        draftee_restriction="season",
        min_minutes=0.0,
    )
    PANEL_PATH = paths.panel_530_csv()
    print("Panel CSV:", PANEL_PATH)
    print("Exists:", PANEL_PATH.is_file())
else:
    print("  CELL 1 skipped  (RUN_CELL1 = False in CELL 0)")

### CELL 2 — Load panel · `perf` · legacy `poolq_loo`

**Data map (Tier1_Briefing_Outline §4):** outcome **`Y_draft`**, own performance **`perf`**, legacy leave-self-out pool quality **`poolq_loo`**. Same `load_panel` + `apply_perf_metric_for_analysis` as **`535` CELL 2** — no extra logic.

In [ ]:
# CELL 2 — Load panel · perf · legacy poolq_loo

if RUN_CELL2:
    if "CFG" not in globals():
        print("  CELL 2 skipped — CFG undefined; run CELL 0 then CELL 1.")
    else:
        import pandas as pd

        from sports_pipeline.panel_build import apply_perf_metric_for_analysis

        df = load_panel(CFG)
        smin = getattr(CFG, "panel_season_min", None)
        smax = getattr(CFG, "panel_season_max", None)
        if smin is not None:
            df = df.loc[pd.to_numeric(df["season"], errors="coerce") >= smin]
        if smax is not None:
            df = df.loc[pd.to_numeric(df["season"], errors="coerce") <= smax]
        df = apply_perf_metric_for_analysis(
            df,
            PERF_METRIC,
            poolq_winsor_quantiles=CFG.poolq_winsor_quantiles,
            zscore_perf_within_season=CFG.perf_zscore_within_season,
        )

        assert "Y_draft" in df.columns, "Panel must include Y_draft"
        assert "poolq_loo" in df.columns
        assert "minutes" in df.columns
else:
    print("  CELL 2 skipped  (RUN_CELL2 = False in CELL 0)")

### CELL 3 — Tier 1 mechanism variables (`congestion_quality` · `congestion_crowding`)

**Data map:** adds Tier 1 **`congestion_quality`** (strict valid-`perf` LOO mean) and crowding columns — same `add_tier1_mechanism_variables` as **`535` CELL 3**.

Sets **`primary_x`**: the default pool column for **\(L\)** in later fit cells, from `PRIMARY_POOL_MODE` / `COMPUTE_WEIGHTED_CROWDING` in CELL 0. (Legacy **`poolq_loo`** remains available for side-by-side specs.)

In [ ]:
# CELL 3 — Tier 1 mechanism variables (congestion_quality · congestion_crowding)

if RUN_CELL3:
    if "df" not in globals():
        print("  CELL 3 skipped — df not defined; run CELL 2 or set RUN_CELL2 = True.")
    else:
        from sports_pipeline.tier1_mechanism_vars import (
            add_tier1_mechanism_variables,
            TIER1_QUALITY_COL,
            TIER1_CROWDING_COL,
            TIER1_CROWDING_WEIGHTED_COL,
            tier1_primary_pool_column,
        )

        df = add_tier1_mechanism_variables(
            df,
            min_minutes=MIN_MINUTES_TIER1,
            minutes_col="minutes",
            perf_col="perf",
            compute_weighted_crowding=COMPUTE_WEIGHTED_CROWDING,
        )

        if PRIMARY_POOL_MODE == "crowding" and COMPUTE_WEIGHTED_CROWDING:
            primary_x = TIER1_CROWDING_WEIGHTED_COL
        else:
            primary_x = tier1_primary_pool_column(PRIMARY_POOL_MODE)

        # Default amalgamated-L proxy for Alex §6 ladder (override in a later cell if needed).
        ALEX_L_COL = primary_x

        print(
            TIER1_QUALITY_COL,
            TIER1_CROWDING_COL,
            TIER1_CROWDING_WEIGHTED_COL,
            "primary_x=",
            primary_x,
            "ALEX_L_COL=",
            ALEX_L_COL,
        )
else:
    print("  CELL 3 skipped  (RUN_CELL3 = False in CELL 0)")

### CELL 4+ *(next)* — Alex §5–§6 fitting ladder

Not yet implemented. Planned flow: **descriptive bins** → **LPM** \(Y \sim L + L^2 + A\) with standardized **`ALEX_L_COL`** and **`perf`** → **logit/probit** → report **\(L^*\)** (mapped back to raw \(Q\) units). Decomposition terms (**`congestion_crowding`**, **`minutes`**) only **one at a time** after the minimal spec. Heavy bin charts stay in **`535` CELL 4**.